# FIG05 helper: candidate single-amino-acid-variant (SAAV) peptide pairs for Skyline.

```
FIG05 helper: candidate single-amino-acid-variant (SAAV) peptide pairs for Skyline.

Finds peptide pairs that are "shared but not quite" between human and bovine: a human-unique
and a bovine-unique peptide that differ by exactly ONE residue (a cross-species SAAV) and
co-elute within ~30 s. These near-identical, co-eluting pairs are what make the two mammals
hard to tell apart -- prime candidates to pull up in Skyline for chromatogram screenshots.

Reimplements the neighbor search in bin/06_homology/peptide_ratios.ipynb (flat human-unique
peptide -> Hamming-distance-1 neighbor), and ADDS the retention-time proximity filter using
DIA-NN's per-precursor RT (report.tsv), which the original did not have.

Algorithm:
  1. Classify each observed peptide by sequence-level homology (in-silico digest; same map as
     FIG05). Keep "Human only" and "Bovine only".
  2. Position-wildcard index -> all human/bovine pairs differing by exactly 1 residue (exact,
     fast: two peptides share a key iff they match everywhere except that one position).
  3. Attach each peptide's median observed RT (report.tsv) and A/C ratio (pr_matrix); keep
     pairs with |dRT| <= ~30 s.
  4. Flag human peptides whose A/C is "flattened" (log2 << expected +3.9) -- the tell-tale of a
     co-eluting bovine SAAV dragging a human-unique peptide toward ratio 1. Sort those first.

Prints the top candidates and writes the full list to output/fig5_saav_candidates.csv.
Input: DIA-NN report.pr_matrix.tsv (A/C) + report.tsv (RT); Combined_proteomes.fasta (classify).
```

In [1]:
import os
import re
import numpy as np
import pandas as pd
from collections import defaultdict

DATA42 = r"D:/2022 Multi-Species Standard Study/42"
PRMATRIX = os.path.join(DATA42, "DIANN_out", "report.pr_matrix.tsv")
REPORT = os.path.join(DATA42, "DIANN_out", "report.tsv")
FASTA = os.path.join(DATA42, "EncyclopeDIA", "Combined_proteomes.fasta")
COLS = {"A": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_A.mzML",
        "C": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_C.mzML"}

OUTPUT = "output"
DATA = "data"
os.makedirs(OUTPUT, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

RT_TOL_MIN = 0.5                       # ~30 s co-elution window
HUMAN_EXPECTED_LOG2 = np.log2(45 / 3)  # +3.91
FLAT_LOG2 = 2.0                        # human A/C "flattened" if log2 < 2 (ratio < 4, exp ~15)
ORGBIT = {"Human": 1, "Bovine": 2, "Trout": 4}
CAT = {1: "Human only", 2: "Bovine only", 4: "Trout only",
       3: "Human+Bovine", 5: "Human+Trout", 6: "Bovine+Trout", 7: "All three"}

In [2]:
def clean_pep(p):
    return re.sub(r"[^A-Z]", "", re.sub(r"\[.*?\]", "", str(p).upper()))

In [3]:
def digest(seq, missed=2, min_len=6, max_len=50):
    seq = re.sub(r"[^A-Z]", "", seq.upper())
    cuts = [0] + [i + 1 for i in range(len(seq) - 1) if seq[i] in "KR" and seq[i + 1] != "P"] + [len(seq)]
    cuts = sorted(set(cuts))
    peps = set()
    for i in range(len(cuts) - 1):
        for m in range(missed + 1):
            j = i + 1 + m
            if j >= len(cuts):
                break
            p = seq[cuts[i]:cuts[j]]
            if min_len <= len(p) <= max_len:
                peps.add(p)
    return peps

In [4]:
def fasta_org(header):
    tok = header.split()[0]
    if "_HUMAN" in tok:
        return "Human"
    if "_BOVIN" in tok:
        return "Bovine"
    return "Trout"

In [5]:
def classify(targets):
    """peptide -> homology category (sequence-level). Shares FIG05's cache."""
    cache = os.path.join(DATA, "fig5_peptide_category.csv")
    if os.path.exists(cache):
        c = pd.read_csv(cache)
        if set(targets).issubset(set(c["peptide"].astype(str))):
            return dict(zip(c["peptide"].astype(str), c["category"]))
    member = {p: 0 for p in targets}

    def flush(header, parts):
        if not header:
            return
        bit = ORGBIT[fasta_org(header)]
        for p in digest("".join(parts)):
            if p in member:
                member[p] |= bit

    header, parts = None, []
    with open(FASTA, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            if line.startswith(">"):
                flush(header, parts)
                header, parts = line[1:].strip(), []
            else:
                parts.append(line.strip())
        flush(header, parts)
    df = pd.DataFrame({"peptide": list(member), "bitmask": list(member.values())})
    df["category"] = df["bitmask"].map(lambda b: CAT.get(b, "Unassigned"))
    df.to_csv(cache, index=False)
    return dict(zip(df["peptide"], df["category"]))

In [6]:
def peptide_table():
    # A/C ratio per peptide (collapse precursors)
    q = pd.read_csv(PRMATRIX, sep="\t", engine="python", on_bad_lines="skip")
    q.columns = [str(c).strip() for c in q.columns]
    q["peptide"] = q["Stripped.Sequence"].map(clean_pep)
    for k, col in COLS.items():
        q[k] = pd.to_numeric(q[col], errors="coerce")
    q = q.groupby("peptide", as_index=False)[["A", "C"]].sum(min_count=1)
    q = q[(q["A"] > 0) & (q["C"] > 0)].copy()
    q["log2_AC"] = np.log2(q["A"] / q["C"])

    # median RT + representative protein per peptide from the main report
    rep = pd.read_csv(REPORT, sep="\t", usecols=["Stripped.Sequence", "RT", "Protein.Ids"], engine="c")
    rep["peptide"] = rep["Stripped.Sequence"].map(clean_pep)
    rt = rep.groupby("peptide")["RT"].median().rename("RT")
    ids = rep.groupby("peptide")["Protein.Ids"].first().rename("protein_ids")

    t = q.merge(rt, on="peptide").merge(ids, on="peptide")
    t["category"] = t["peptide"].map(classify(set(t["peptide"])))
    return t

In [7]:
def find_saav_pairs(t):
    hum = t[t["category"] == "Human only"].set_index("peptide")
    bov = t[t["category"] == "Bovine only"].set_index("peptide")

    # position-wildcard index of bovine peptides: key seq with one position -> "*"
    idx = defaultdict(list)
    for pep in bov.index:
        for i in range(len(pep)):
            idx[pep[:i] + "*" + pep[i + 1:]].append(pep)

    rows = []
    for hpep in hum.index:
        for i in range(len(hpep)):
            for bpep in idx.get(hpep[:i] + "*" + hpep[i + 1:], []):
                if hpep == bpep:
                    continue
                hr, br = hum.loc[hpep], bov.loc[bpep]
                drt = abs(hr["RT"] - br["RT"])
                if drt <= RT_TOL_MIN:
                    rows.append(dict(
                        human_peptide=hpep, bovine_peptide=bpep,
                        substitution=f"{i+1}:{hpep[i]}>{bpep[i]}",
                        dRT_sec=round(drt * 60, 1),
                        human_RT=round(hr["RT"], 2), bovine_RT=round(br["RT"], 2),
                        human_log2AC=round(hr["log2_AC"], 2), bovine_log2AC=round(br["log2_AC"], 2),
                        human_flattened=bool(hr["log2_AC"] < FLAT_LOG2),
                        human_ids=hr["protein_ids"], bovine_ids=br["protein_ids"]))
    df = pd.DataFrame(rows).drop_duplicates(subset=["human_peptide", "bovine_peptide"])
    # best Skyline candidates first: flattened human, then closest co-elution
    return df.sort_values(["human_flattened", "dRT_sec"], ascending=[False, True]).reset_index(drop=True)

In [8]:
# ---- find + print SAAV candidates ----
t = peptide_table()
pairs = find_saav_pairs(t)
out_csv = os.path.join(OUTPUT, "fig5_saav_candidates.csv")
pairs.to_csv(out_csv, index=False)
print(f"human-unique={sum(t.category=='Human only')}, bovine-unique={sum(t.category=='Bovine only')}")
print(f"{len(pairs)} human/bovine SAAV pairs (1 aa apart) co-eluting within {RT_TOL_MIN*60:.0f} s")
print(f"  ({pairs['human_flattened'].sum()} have a flattened human ratio -> best Skyline examples)")
print(f"  full list -> {out_csv}\n")
show = ["human_peptide", "bovine_peptide", "substitution", "dRT_sec",
        "human_log2AC", "bovine_log2AC", "human_flattened"]
with pd.option_context("display.width", 200, "display.max_colwidth", 30):
    print(pairs[show].head(25).to_string(index=False))

human-unique=1452, bovine-unique=2409
67 human/bovine SAAV pairs (1 aa apart) co-eluting within 30 s
  (7 have a flattened human ratio -> best Skyline examples)
  full list -> output\fig5_saav_candidates.csv

               human_peptide               bovine_peptide substitution  dRT_sec  human_log2AC  bovine_log2AC  human_flattened
         ELGLREENEGVYNGSWGGR          ELGLREENDGVYNGSWGGR        9:E>D      2.0          0.99          -3.23             True
  QLEAEKMELQSALEEAEASLEHEEGK   QLEAEKLELQSALEEAEASLEHEEGK        7:M>L      3.6         -0.67          -0.68             True
                AMLSTGFKIPQK                 AMLSTGFKLPQK        9:I>L     12.9          0.65           1.19             True
             ALSQHPTLNDDLPNR              ALSQHPTINDDLPNR        8:L>I     13.7          0.35           2.26             True
             INLPAPNPDHVGGYK              ITLPAPNPDHVGGYK        2:N>T     13.9         -1.17          -3.41             True
              LLLEFTDTSYEEKR       